# Engenharia de Features (v6)
## v4 + Histórico de Copas Anteriores

**Motivação:** As features de histórico de Copas foram calculadas na v5 mas nunca entraram no modelo final porque a v5 também adicionou valor de mercado, reduzindo as amostras de treino de 216 para 64. A v6 corrige isso — adiciona apenas o histórico, que está disponível para **todas as Copas de 1994 a 2022**, mantendo as 216 amostras.

**Features novas:**
- `media_gols_ultimas2_copas` — média de gols nas 2 Copas anteriores (0 se não participou)
- `fase_ultima_copa` — fase atingida na Copa imediatamente anterior (0=não classificou, 1=grupos, 2=oitavas, 3=quartas, 4=semi, 5=final, 6=campeão)

**Dataset de saída:** `data/processed/features_completo_v6.csv`

## 1. Imports e Carregamento

In [ ]:
import pandas as pd
import numpy as np
import sys
sys.path.append('../src/features')
from elo import calcular_elo_historico

df_raw = pd.read_csv('../data/raw/results.csv', parse_dates=['date'])
print(f'Shape: {df_raw.shape}')

## 2. Calculando ELO Histórico

In [ ]:
df = calcular_elo_historico(df_raw, elo_inicial=1000, k_competitivo=40, k_amistoso=20)
print(f'ELO calculado para {len(df)} jogos')

## 3. Funções de Histórico de Copas

In [ ]:
copas_datas = {
    1990: ('1990-06-08', '1990-07-08'),
    1994: ('1994-06-17', '1994-07-17'),
    1998: ('1998-06-10', '1998-07-12'),
    2002: ('2002-05-31', '2002-06-30'),
    2006: ('2006-06-09', '2006-07-09'),
    2010: ('2010-06-11', '2010-07-11'),
    2014: ('2014-06-12', '2014-07-13'),
    2018: ('2018-06-14', '2018-07-15'),
    2022: ('2022-11-20', '2022-12-18'),
}

def get_jogos_copa(df, ano):
    inicio, fim = copas_datas[ano]
    return df[
        (df['tournament'] == 'FIFA World Cup') &
        (df['date'] >= inicio) &
        (df['date'] <= fim)
    ]

def get_fase(df_copa, selecao):
    jogos = df_copa[
        (df_copa['home_team'] == selecao) |
        (df_copa['away_team'] == selecao)
    ]
    if len(jogos) == 0:
        return 0
    n = len(jogos)
    if n == 3:   return 1  # grupos
    elif n == 4: return 2  # oitavas
    elif n == 5: return 3  # quartas
    elif n == 6: return 4  # semi
    elif n == 7:
        ultimo = jogos.sort_values('date').iloc[-1]
        if ultimo['home_team'] == selecao:
            ganhou = ultimo['home_score'] > ultimo['away_score']
        else:
            ganhou = ultimo['away_score'] > ultimo['home_score']
        return 6 if ganhou else 5
    else:
        return min(n - 2, 6)

def get_media_gols_copa(df_copa, selecao):
    jogos = df_copa[
        (df_copa['home_team'] == selecao) |
        (df_copa['away_team'] == selecao)
    ]
    if len(jogos) == 0:
        return np.nan
    gols = [
        row['home_score'] if row['home_team'] == selecao else row['away_score']
        for _, row in jogos.iterrows()
    ]
    return np.mean(gols)

print('Funções definidas!')

# Validação
copa22 = get_jogos_copa(df, 2022)
copa18 = get_jogos_copa(df, 2018)
print(f'\nValidação:')
print(f'  França fase 2018 (esperado 6-campeã): {get_fase(copa18, "France")}')
print(f'  Bélgica fase 2018 (esperado 4-semi):  {get_fase(copa18, "Belgium")}')
print(f'  Holanda fase 2018 (esperado 0-fora):  {get_fase(copa18, "Netherlands")}')
print(f'  Brasil fase 2018 (esperado 3-quartas):{get_fase(copa18, "Brazil")}')

## 4. Função de Features v6

In [ ]:
def calcular_features_v6(selecao, ciclo, copa_df, copa_ano, min_jogos=15):
    """
    Features v4 (9) + histórico de Copas (2) = 11 features.
    Mantém 216 amostras de treino.
    """
    # --- Features v4 ---
    jogos = ciclo[
        (ciclo['home_team'] == selecao) |
        (ciclo['away_team'] == selecao)
    ].sort_values('date')

    if len(jogos) < min_jogos:
        raise ValueError(f'Apenas {len(jogos)} jogos')

    gm, gs, vit, elo_adv = [], [], [], []
    for _, row in jogos.iterrows():
        if row['home_team'] == selecao:
            gm.append(row['home_score']); gs.append(row['away_score'])
            vit.append(1 if row['home_score'] > row['away_score'] else 0)
            elo_adv.append(row['elo_away_antes'])
        else:
            gm.append(row['away_score']); gs.append(row['home_score'])
            vit.append(1 if row['away_score'] > row['home_score'] else 0)
            elo_adv.append(row['elo_home_antes'])

    gm, gs, vit = np.array(gm), np.array(gs), np.array(vit)
    elo_adv = np.array(elo_adv)

    ult15 = jogos.tail(15)
    gm15, gs15, vit15, ea15 = [], [], [], []
    for _, row in ult15.iterrows():
        if row['home_team'] == selecao:
            gm15.append(row['home_score']); gs15.append(row['away_score'])
            vit15.append(1 if row['home_score'] > row['away_score'] else 0)
            ea15.append(row['elo_away_antes'])
        else:
            gm15.append(row['away_score']); gs15.append(row['home_score'])
            vit15.append(1 if row['away_score'] > row['home_score'] else 0)
            ea15.append(row['elo_home_antes'])

    # Target
    sel_copa = copa_df[
        (copa_df['home_team'] == selecao) |
        (copa_df['away_team'] == selecao)
    ]
    gols_copa = [
        row['home_score'] if row['home_team'] == selecao else row['away_score']
        for _, row in sel_copa.iterrows()
    ]

    # --- Features v6 novas: histórico de Copas ---
    anos_copas = sorted(copas_datas.keys())
    idx_atual  = anos_copas.index(copa_ano)

    # Média de gols nas últimas 2 Copas
    copas_anteriores = anos_copas[max(0, idx_atual-2):idx_atual]
    gols_hist = []
    for ano_h in copas_anteriores:
        inicio_h, fim_h = copas_datas[ano_h]
        copa_h = df[
            (df['tournament'] == 'FIFA World Cup') &
            (df['date'] >= inicio_h) & (df['date'] <= fim_h)
        ]
        mg = get_media_gols_copa(copa_h, selecao)
        if not np.isnan(mg):
            gols_hist.append(mg)
    media_gols_hist = np.mean(gols_hist) if gols_hist else 0.0

    # Fase da Copa anterior
    if idx_atual > 0:
        ano_ant = anos_copas[idx_atual - 1]
        inicio_ant, fim_ant = copas_datas[ano_ant]
        copa_ant = df[
            (df['tournament'] == 'FIFA World Cup') &
            (df['date'] >= inicio_ant) & (df['date'] <= fim_ant)
        ]
        fase_ant = get_fase(copa_ant, selecao)
    else:
        fase_ant = 0

    return {
        # v4
        'media_gols_marcados_ciclo': gm.mean(),
        'media_gols_sofridos_ciclo': gs.mean(),
        'pct_vitorias_ciclo':        vit.mean(),
        'total_jogos_ciclo':         len(jogos),
        'media_gols_marcados_ult15': np.array(gm15).mean(),
        'media_gols_sofridos_ult15': np.array(gs15).mean(),
        'pct_vitorias_ult15':        np.array(vit15).mean(),
        'elo_medio_adv_ciclo':       elo_adv.mean(),
        'elo_medio_adv_ult15':       np.array(ea15).mean(),
        # v6 novas
        'media_gols_ultimas2_copas': media_gols_hist,
        'fase_ultima_copa':          fase_ant,
        # target
        'media_gols_copa':           np.mean(gols_copa)
    }

print('Função v6 definida!')

# Teste com Brasil 2022
copa_2022 = df[
    (df['tournament'] == 'FIFA World Cup') &
    (df['date'] >= '2022-11-20') & (df['date'] <= '2022-12-18')
]
ciclo_2022 = df[
    (df['date'] >= '2018-07-16') &
    (df['date'] <= '2022-11-19') &
    (df['tournament'] != 'FIFA World Cup')
]
brasil = calcular_features_v6('Brazil', ciclo_2022, copa_2022, 2022)
print('\nTeste Brasil 2022:')
for k, v in brasil.items():
    print(f'  {k:<35} {v:.4f}')

## 5. Pipeline Completo — Todas as Copas (1994–2022)

In [ ]:
copas = {
    1994: {'ciclo_inicio': '1990-07-09', 'ciclo_fim': '1994-06-16', 'copa_inicio': '1994-06-17', 'copa_fim': '1994-07-17'},
    1998: {'ciclo_inicio': '1994-07-18', 'ciclo_fim': '1998-06-09', 'copa_inicio': '1998-06-10', 'copa_fim': '1998-07-12'},
    2002: {'ciclo_inicio': '1998-07-13', 'ciclo_fim': '2002-05-30', 'copa_inicio': '2002-05-31', 'copa_fim': '2002-06-30'},
    2006: {'ciclo_inicio': '2002-07-01', 'ciclo_fim': '2006-06-08', 'copa_inicio': '2006-06-09', 'copa_fim': '2006-07-09'},
    2010: {'ciclo_inicio': '2006-07-10', 'ciclo_fim': '2010-06-10', 'copa_inicio': '2010-06-11', 'copa_fim': '2010-07-11'},
    2014: {'ciclo_inicio': '2010-07-12', 'ciclo_fim': '2014-06-11', 'copa_inicio': '2014-06-12', 'copa_fim': '2014-07-13'},
    2018: {'ciclo_inicio': '2014-07-14', 'ciclo_fim': '2018-06-13', 'copa_inicio': '2018-06-14', 'copa_fim': '2018-07-15'},
    2022: {'ciclo_inicio': '2018-07-16', 'ciclo_fim': '2022-11-19', 'copa_inicio': '2022-11-20', 'copa_fim': '2022-12-18'},
}

todos_dados = []
filtrados   = []

for ano, datas in copas.items():
    print(f'Processando Copa {ano}...')

    ciclo = df[
        (df['date'] >= datas['ciclo_inicio']) &
        (df['date'] <= datas['ciclo_fim']) &
        (df['tournament'] != 'FIFA World Cup')
    ]
    copa_df = df[
        (df['tournament'] == 'FIFA World Cup') &
        (df['date'] >= datas['copa_inicio']) &
        (df['date'] <= datas['copa_fim'])
    ]

    selecoes = pd.unique(copa_df[['home_team', 'away_team']].values.ravel())

    for selecao in selecoes:
        try:
            resultado = calcular_features_v6(selecao, ciclo, copa_df, ano)
            resultado['selecao']   = selecao
            resultado['copa_alvo'] = ano
            todos_dados.append(resultado)
        except ValueError as e:
            filtrados.append({'selecao': selecao, 'copa': ano, 'motivo': str(e)})
        except Exception as e:
            print(f'  Erro em {selecao}: {e}')

df_v6 = pd.DataFrame(todos_dados)
print(f'\nDataset v6: {df_v6.shape[0]} linhas × {df_v6.shape[1]} colunas')
print(f'Filtrados: {len(filtrados)}')
print(f'\nLinhas por Copa:')
print(df_v6['copa_alvo'].value_counts().sort_index())

## 6. Validação das Features Históricas

In [ ]:
print('Histórico Copa 2022 — Top 10 por fase anterior:')
cols = ['selecao', 'fase_ultima_copa', 'media_gols_ultimas2_copas', 'media_gols_copa']
print(df_v6[df_v6['copa_alvo'] == 2022][cols]
      .sort_values('fase_ultima_copa', ascending=False)
      .head(10).to_string(index=False))

# Correlação das novas features com o target
corr_fase  = df_v6['fase_ultima_copa'].corr(df_v6['media_gols_copa'])
corr_hist  = df_v6['media_gols_ultimas2_copas'].corr(df_v6['media_gols_copa'])
print(f'\nCorrelação fase_ultima_copa vs target:           {corr_fase:.4f}')
print(f'Correlação media_gols_ultimas2_copas vs target:  {corr_hist:.4f}')

## 7. Salvamento

In [ ]:
df_v6.to_csv('../data/processed/features_completo_v6.csv', index=False)
print('Dataset v6 salvo em: data/processed/features_completo_v6.csv')

## 8. Resumo das Features

| Feature | Versão | Descrição |
|---------|--------|-----------|
| 7 features originais | v1 | ciclo + últimos 15 |
| `elo_medio_adv_ciclo/ult15` | v4 | ELO médio dos adversários |
| `media_gols_ultimas2_copas` | **v6** | histórico ofensivo nas Copas |
| `fase_ultima_copa` | **v6** | desempenho na Copa anterior |

**Vantagem sobre v5:** mantém 216 amostras de treino — sem perda de dados.

**Próximo passo:** `03_modelos_v6.ipynb` — comparar v4 vs v6.